# Fake news detection

Building a model to classify news articles as real or fake using a LinearSVC classifier.

In [76]:
import pandas as pd

# Importing datasets

df_fake = pd.read_csv("data/Fake.csv")
df_true = pd.read_csv("data/True.csv")

# Check that works for both dataframes
df_fake.head()



,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [77]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [78]:
# Inserting column "class" as target feature

df_fake["class"] = 0
df_true["class"] = 1


# Match sizes by truncating the larger DataFrame
min_rows = min(len(df_fake), len(df_true))
df_fake = df_fake.iloc[:min_rows, :]
df_true = df_true.iloc[:min_rows, :]


print(df_fake.shape, df_true.shape)


(21417, 5) (21417, 5)


In [79]:
# Merging dataframes
df_both = pd.concat([df_fake, df_true], axis=0)

# Remove columns that we don't need, keeping title, text and class
df = df_both.drop(["subject", "date", "title"], axis=1)

df.head()

,text,class
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [80]:
# Checking data integrity
df.isnull().sum()


text     0
class    0
dtype: int64

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.svm import LinearSVC

X, y = df["text"], df["class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Vectorizing text data
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.7)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Creating the classifier
clf = LinearSVC()
clf.fit(X_train_vectorized, y_train)

# Checking accuracy
accuracy = clf.score(X_test_vectorized, y_test) 
print(f"Accuracy: {accuracy:.2f}")

# Predictions
y_pred = clf.predict(X_test_vectorized)

# Classification report
cr = classification_report(y_test, y_pred)
print("\nClassification Report:\n", cr)

# accuracy
accuracy_explicit = accuracy_score(y_test, y_pred)
print(f"Explicit Accuracy: {accuracy_explicit:.2f}")

Accuracy: 0.99

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99      4241
           1       0.99      1.00      0.99      4326

    accuracy                           0.99      8567
   macro avg       0.99      0.99      0.99      8567
weighted avg       0.99      0.99      0.99      8567

Explicit Accuracy: 0.99


In [82]:
print(len(y_test) * 0.9950974670246294, "correct out of", len(y_test))

8525.0 correct out of 8567


In [83]:
# Checking performance with cross-validation

from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X_train_vectorized, y_train, cv=5)
print(f"Cross-Validation Scores: {scores}")
print(f"Average Cross-Validation Score: {scores.mean():.2f}")


Cross-Validation Scores: [0.99197549 0.99182959 0.98920181 0.99226616 0.99284985]
Average Cross-Validation Score: 0.99
